In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
from ALLCools.mcds import MCDS
from ALLCools.clustering import tsne, significant_pc_test, filter_regions, lsi, binarize_matrix
from ALLCools.plot import *

In [ ]:
metadata_path = 'CellMetadata.PassQC.Quintile.correctCellNames.csv.gz'
mcds_path = './GSC_epitherapy_with5k.mcds/'

# Basic filtering parameters
mapping_rate_cutoff = 0.5
mapping_rate_col_name = 'R1MappingRate'  # Name may change
final_reads_cutoff = 500000
final_reads_col_name = 'FinalmCReads'  # Name may change
mccc_cutoff = 0.03
mccc_col_name = 'mCCCFrac'  # Name may change
mch_cutoff = 0.2
mch_col_name = 'mCHFrac'  # Name may change
mcg_cutoff = 0.05
mcg_col_name = 'mCGFrac'  # Name may change

# PC cutoff
pc_cutoff = 0.1

# KNN
knn = -1  # -1 means auto determine

# Leiden
resolution = 1

In [ ]:
metadata = pd.read_csv(metadata_path, index_col=0)
print(f'Metadata of {metadata.shape[0]} cells')
metadata.head()

In [ ]:
judge = (metadata[mapping_rate_col_name] > mapping_rate_cutoff) & \
        (metadata[final_reads_col_name] > final_reads_cutoff) & \
        (metadata[mccc_col_name] < mccc_cutoff) & \
        (metadata[mch_col_name] < mch_cutoff) & \
        (metadata[mcg_col_name] > mcg_cutoff)

metadata = metadata[judge].copy()
# cell metadata for this example is filtered already
print(f'{metadata.shape[0]} cells passed filtering')

In [ ]:
mcds = MCDS.open(mcds_path, obs_dim='cell', var_dim='chrom5k', use_obs=metadata.index)

In [ ]:
mcds.add_cell_metadata(metadata)

In [ ]:
mcds = mcds.remove_black_list_region(black_list_path='/scratch/devtools/mmoore/genomes/snm3c/hg38/hg38-blacklist.v2.bed.gz')

In [ ]:
mcad = mcds.get_score_adata(mc_type='CGN', quant_type='hypo-score')

In [ ]:
mcad

In [ ]:
binarize_matrix(mcad, cutoff=0.95)

In [ ]:
filter_regions(mcad)

In [ ]:
mcad

In [ ]:
# by default we save the results in adata.obsm['X_pca'] which is the scanpy defaults in many following functions
# But this matrix is not calculated by PCA
lsi(mcad, algorithm='arpack', obsm='X_pca')

In [ ]:
# choose significant components
significant_pc_test(mcad, p_cutoff=pc_cutoff, update=True)

In [ ]:
if knn == -1:
    knn = max(15, int(np.log2(mcad.shape[0])*2))
sc.pp.neighbors(mcad, n_neighbors=knn)

In [ ]:
sc.tl.leiden(mcad, resolution=resolution)

In [ ]:
tsne(mcad,
     obsm='X_pca',
     metric='euclidean',
     exaggeration=-1,  # auto determined
     perplexity=30,
     n_jobs=-1)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
_ = categorical_scatter(data=mcad,
                        ax=ax,
                        coord_base='tsne',
                        hue='Cell_line',
                        text_anno='leiden',
                        show_legend=True)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
_ = categorical_scatter(data=mcad,
                        ax=ax,
                        coord_base='tsne',
                        hue='Treatment',
                        text_anno='leiden',
                        show_legend=True)

In [ ]:
sc.tl.umap(mcad)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
_ = categorical_scatter(data=mcad,
                        ax=ax,
                        coord_base='umap',
                        hue='Cell_line',
                        #text_anno='leiden',
                        show_legend=True)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
_ = categorical_scatter(data=mcad,
                        ax=ax,
                        coord_base='umap',
                        hue='Treatment',
                        #text_anno='leiden',
                        show_legend=True)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
_ = categorical_scatter(data=mcad,
                        ax=ax,
                        coord_base='umap',
                        hue='qcluster',
                        text_anno='leiden',
                        show_legend=True)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=300)
_ = continuous_scatter(data=mcad,
                        ax=ax,
                        coord_base='umap',
                        hue='mCGFrac')
                        #text_anno='leiden',
                        #show_legend=True)

In [ ]:
interactive_scatter(data=mcad, hue='leiden', coord_base='umap')

In [ ]:
mcad.write_h5ad('GSC.mCG-5K-clustering.h5ad')
mcad

In [ ]:
mcad.obs.to_csv('GSC.ClusteringResults5k.csv.gz')
mcad.obs.head()

In [ ]:
mcad.obsm["X_umap"]

In [ ]:
umap = mcad.obsm["X_umap"]

print(umap.shape)

In [ ]:
from scipy.spatial.distance import pdist, squareform

dist_matrix = squareform(pdist(umap, metric="euclidean"))

print(dist_matrix.shape)

In [ ]:
import seaborn as sns
g = sns.clustermap(
    dist_matrix,
    cmap="viridis",
    row_cluster=True,
    col_cluster=True,
    #row_colors=row_colors,
    #col_colors=row_colors,
    figsize=(10,10)
)

In [ ]:
dist_df = pd.DataFrame(dist_matrix, index=mcad.obs_names.values, columns=mcad.obs_names.values)
dist_df.to_csv("cell_cell_euclidean.tsv", sep="\t")

In [ ]:
mcad.obs_names.values